In [1]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# =========================
# 경로 설정
# =========================
INPUT_DIR = r"C:\Users\Konyang\Desktop\cropped_dataset_more"
OUTPUT_DIR = r"C:\Users\Konyang\Desktop\cropped_preprocessed_dataset"

# =========================
# 출력 폴더 생성
# =========================
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# Gamma Correction 함수
# =========================
def gamma_correction(image, gamma=1.1):
    inv_gamma = 1.0 / gamma

    table = np.array([
        ((i / 255.0) ** inv_gamma) * 255
        for i in np.arange(256)
    ]).astype("uint8")

    return cv2.LUT(image, table)

# =========================
# CLAHE 함수
# =========================
def apply_clahe(image):
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)

    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    cl = clahe.apply(l)

    merged = cv2.merge((cl, a, b))

    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

# =========================
# 이미지 전처리
# =========================
image_extensions = (".jpg", ".jpeg", ".png", ".bmp")

image_files = []

for root, dirs, files in os.walk(INPUT_DIR):
    for file in files:
        if file.lower().endswith(image_extensions):
            image_files.append(os.path.join(root, file))

print(f"총 이미지 수: {len(image_files)}")

for img_path in tqdm(image_files):

    # 이미지 읽기
    image = cv2.imread(img_path)

    if image is None:
        continue

    # =========================
    # 1. Resize (320x320)
    # =========================
    image = cv2.resize(image, (320, 320))

    # =========================
    # 2. Gaussian Blur (3x3)
    # =========================
    image = cv2.GaussianBlur(image, (3, 3), 0)

    # =========================
    # 3. CLAHE
    # =========================
    image = apply_clahe(image)

    # =========================
    # 4. Gamma Correction (1.1)
    # =========================
    image = gamma_correction(image, gamma=1.1)

    # =========================
    # 저장 경로 생성
    # =========================
    relative_path = os.path.relpath(img_path, INPUT_DIR)

    save_path = os.path.join(OUTPUT_DIR, relative_path)

    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    # 저장
    cv2.imwrite(save_path, image)

print("\n전처리 완료")
print("저장 경로:", OUTPUT_DIR)

총 이미지 수: 17988


100%|███████████████████████████████████████████████████████████████████████████| 17988/17988 [01:31<00:00, 195.69it/s]


전처리 완료
저장 경로: C:\Users\Konyang\Desktop\cropped_preprocessed_dataset


In [2]:
import os
import hashlib
from collections import defaultdict

# =========================
# 데이터 경로
# =========================
DATA_ROOT = r"C:\Users\Konyang\Desktop\cropped_preprocessed_dataset"

train_dir = os.path.join(DATA_ROOT, "train")
val_dir = os.path.join(DATA_ROOT, "val")
test_dir = os.path.join(DATA_ROOT, "test")

# =========================
# 이미지 확장자
# =========================
IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

# =========================
# 파일 해시 생성
# =========================
def get_file_hash(file_path):
    hasher = hashlib.md5()

    with open(file_path, "rb") as f:
        while chunk := f.read(8192):
            hasher.update(chunk)

    return hasher.hexdigest()

# =========================
# 이미지 수집
# =========================
def collect_images(root_dir):
    image_dict = {}

    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.lower().endswith(IMAGE_EXTS):

                full_path = os.path.join(root, file)

                try:
                    file_hash = get_file_hash(full_path)
                    image_dict[file_hash] = full_path

                except Exception as e:
                    print(f"오류: {full_path}")
                    print(e)

    return image_dict

# =========================
# 데이터 읽기
# =========================
print("=" * 70)
print("이미지 수집 시작")
print("=" * 70)

train_images = collect_images(train_dir)
val_images = collect_images(val_dir)
test_images = collect_images(test_dir)

# =========================
# 중복 검사
# =========================
train_val_overlap = set(train_images.keys()) & set(val_images.keys())
train_test_overlap = set(train_images.keys()) & set(test_images.keys())
val_test_overlap = set(val_images.keys()) & set(test_images.keys())

# =========================
# 결과 출력
# =========================
print("\n" + "=" * 70)
print("중복 이미지 검사 결과")
print("=" * 70)

print(f"Train 이미지 수 : {len(train_images)}")
print(f"Val 이미지 수   : {len(val_images)}")
print(f"Test 이미지 수  : {len(test_images)}")

print("\n[Train ↔ Val 중복]")
print("중복 개수 :", len(train_val_overlap))

print("\n[Train ↔ Test 중복]")
print("중복 개수 :", len(train_test_overlap))

print("\n[Val ↔ Test 중복]")
print("중복 개수 :", len(val_test_overlap))

# =========================
# 중복 파일 일부 출력
# =========================
def print_overlap_examples(overlap_set, dict1, title):
    print(f"\n{title}")

    if len(overlap_set) == 0:
        print("중복 없음")
        return

    for i, h in enumerate(list(overlap_set)[:10]):
        print(dict1[h])

print_overlap_examples(
    train_val_overlap,
    train_images,
    "[Train ↔ Val 중복 예시]"
)

print_overlap_examples(
    train_test_overlap,
    train_images,
    "[Train ↔ Test 중복 예시]"
)

print_overlap_examples(
    val_test_overlap,
    val_images,
    "[Val ↔ Test 중복 예시]"
)

print("\n검사 완료")

이미지 수집 시작

중복 이미지 검사 결과
Train 이미지 수 : 14393
Val 이미지 수   : 1777
Test 이미지 수  : 1818

[Train ↔ Val 중복]
중복 개수 : 0

[Train ↔ Test 중복]
중복 개수 : 0

[Val ↔ Test 중복]
중복 개수 : 0

[Train ↔ Val 중복 예시]
중복 없음

[Train ↔ Test 중복 예시]
중복 없음

[Val ↔ Test 중복 예시]
중복 없음

검사 완료


In [3]:
import os
import hashlib
from collections import Counter, defaultdict
from torchvision import datasets, transforms, models
import torch
import torch.nn as nn

# =========================
# 경로 설정
# =========================
DATA_ROOT = r"C:\Users\Konyang\Desktop\cropped_preprocessed_dataset"

train_dir = os.path.join(DATA_ROOT, "train")
val_dir = os.path.join(DATA_ROOT, "val")
test_dir = os.path.join(DATA_ROOT, "test")

IMAGE_EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".webp")

# =========================
# 기본 폴더 확인
# =========================
print("=" * 70)
print("1. 기본 경로 확인")
print("=" * 70)

for name, path in [("train", train_dir), ("val", val_dir), ("test", test_dir)]:
    print(f"{name} 경로: {path}")
    print(f"{name} 존재 여부: {os.path.exists(path)}")
    print()

# =========================
# 폴더 구조 확인
# =========================
print("=" * 70)
print("2. train/val/test 내부 폴더 확인")
print("=" * 70)

for split_name, split_dir in [("train", train_dir), ("val", val_dir), ("test", test_dir)]:
    print(f"\n[{split_name}]")

    if not os.path.exists(split_dir):
        print("폴더 없음")
        continue

    folders = [
        f for f in os.listdir(split_dir)
        if os.path.isdir(os.path.join(split_dir, f))
    ]

    files = [
        f for f in os.listdir(split_dir)
        if os.path.isfile(os.path.join(split_dir, f))
    ]

    print("클래스로 인식될 폴더:")
    print(folders)

    if files:
        print("주의: split 폴더 바로 아래에 파일 존재")
        print(files[:10])

# =========================
# ImageFolder로 읽기
# =========================
print("\n" + "=" * 70)
print("3. ImageFolder 클래스 매핑 확인")
print("=" * 70)

basic_transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(train_dir, transform=basic_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=basic_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=basic_transform)

print("train class_to_idx:", train_dataset.class_to_idx)
print("val   class_to_idx:", val_dataset.class_to_idx)
print("test  class_to_idx:", test_dataset.class_to_idx)

print("\ntrain classes:", train_dataset.classes)
print("val   classes:", val_dataset.classes)
print("test  classes:", test_dataset.classes)

print("\nnum_classes")
print("train:", len(train_dataset.classes))
print("val  :", len(val_dataset.classes))
print("test :", len(test_dataset.classes))

# =========================
# 클래스 매핑 일치 여부
# =========================
print("\n" + "=" * 70)
print("4. 클래스 매핑 일치 여부")
print("=" * 70)

if train_dataset.class_to_idx == val_dataset.class_to_idx == test_dataset.class_to_idx:
    print("정상: train/val/test 클래스 매핑이 모두 동일함")
else:
    print("문제: train/val/test 클래스 매핑이 서로 다름")
    print("이 경우 평가 결과가 이상하게 나올 수 있음")

# =========================
# 이미지 개수 / 클래스 분포 확인
# =========================
print("\n" + "=" * 70)
print("5. 클래스별 이미지 개수 확인")
print("=" * 70)

def print_class_distribution(dataset, split_name):
    counter = Counter([label for _, label in dataset.samples])
    idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}

    print(f"\n[{split_name}] 총 이미지 수: {len(dataset)}")

    for idx in sorted(counter.keys()):
        print(f"{idx_to_class[idx]}: {counter[idx]}장")

print_class_distribution(train_dataset, "train")
print_class_distribution(val_dataset, "val")
print_class_distribution(test_dataset, "test")

# =========================
# 샘플 경로와 라벨 확인
# =========================
print("\n" + "=" * 70)
print("6. 샘플 이미지 경로 / 라벨 확인")
print("=" * 70)

def print_samples(dataset, split_name, n=10):
    idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}

    print(f"\n[{split_name}] 샘플 {n}개")
    for path, label in dataset.samples[:n]:
        print(f"label={label} ({idx_to_class[label]}) | {path}")

print_samples(train_dataset, "train")
print_samples(val_dataset, "val")
print_samples(test_dataset, "test")

# =========================
# 같은 파일명 중복 검사
# =========================
print("\n" + "=" * 70)
print("7. train/val/test 같은 파일명 중복 검사")
print("=" * 70)

def collect_filenames(split_dir):
    result = defaultdict(list)

    for root, dirs, files in os.walk(split_dir):
        for file in files:
            if file.lower().endswith(IMAGE_EXTS):
                result[file].append(os.path.join(root, file))

    return result

train_names = collect_filenames(train_dir)
val_names = collect_filenames(val_dir)
test_names = collect_filenames(test_dir)

train_val_name_overlap = set(train_names.keys()) & set(val_names.keys())
train_test_name_overlap = set(train_names.keys()) & set(test_names.keys())
val_test_name_overlap = set(val_names.keys()) & set(test_names.keys())

print("Train ↔ Val 같은 파일명:", len(train_val_name_overlap))
print("Train ↔ Test 같은 파일명:", len(train_test_name_overlap))
print("Val ↔ Test 같은 파일명:", len(val_test_name_overlap))

if train_val_name_overlap:
    print("\nTrain ↔ Val 같은 파일명 예시:")
    for name in list(train_val_name_overlap)[:10]:
        print(name)

if train_test_name_overlap:
    print("\nTrain ↔ Test 같은 파일명 예시:")
    for name in list(train_test_name_overlap)[:10]:
        print(name)

if val_test_name_overlap:
    print("\nVal ↔ Test 같은 파일명 예시:")
    for name in list(val_test_name_overlap)[:10]:
        print(name)

# =========================
# 파일 해시 중복 검사
# =========================
print("\n" + "=" * 70)
print("8. 실제 이미지 내용 중복 검사")
print("=" * 70)

def get_file_hash(file_path):
    hasher = hashlib.md5()

    with open(file_path, "rb") as f:
        while True:
            chunk = f.read(8192)
            if not chunk:
                break
            hasher.update(chunk)

    return hasher.hexdigest()

def collect_hashes(split_dir):
    result = {}

    for root, dirs, files in os.walk(split_dir):
        for file in files:
            if file.lower().endswith(IMAGE_EXTS):
                path = os.path.join(root, file)

                try:
                    h = get_file_hash(path)
                    result[h] = path
                except Exception as e:
                    print("해시 오류:", path, e)

    return result

train_hashes = collect_hashes(train_dir)
val_hashes = collect_hashes(val_dir)
test_hashes = collect_hashes(test_dir)

train_val_hash_overlap = set(train_hashes.keys()) & set(val_hashes.keys())
train_test_hash_overlap = set(train_hashes.keys()) & set(test_hashes.keys())
val_test_hash_overlap = set(val_hashes.keys()) & set(test_hashes.keys())

print("Train ↔ Val 실제 중복:", len(train_val_hash_overlap))
print("Train ↔ Test 실제 중복:", len(train_test_hash_overlap))
print("Val ↔ Test 실제 중복:", len(val_test_hash_overlap))

# =========================
# 모델 출력 클래스 수 확인
# =========================
print("\n" + "=" * 70)
print("9. ConvNeXt-Tiny 출력 클래스 수 확인")
print("=" * 70)

num_classes = len(train_dataset.classes)

model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)

in_features = model.classifier[2].in_features
model.classifier[2] = nn.Linear(in_features, num_classes)

print(model.classifier)
print("\n마지막 Linear 확인:")
print(model.classifier[2])

# =========================
# 배치 하나 확인
# =========================
print("\n" + "=" * 70)
print("10. DataLoader 배치 라벨 확인")
print("=" * 70)

loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=0
)

images, labels = next(iter(loader))

print("images shape:", images.shape)
print("labels:", labels.tolist())
print("labels unique:", sorted(set(labels.tolist())))

print("\n검사 완료")

1. 기본 경로 확인
train 경로: C:\Users\Konyang\Desktop\cropped_preprocessed_dataset\train
train 존재 여부: True

val 경로: C:\Users\Konyang\Desktop\cropped_preprocessed_dataset\val
val 존재 여부: True

test 경로: C:\Users\Konyang\Desktop\cropped_preprocessed_dataset\test
test 존재 여부: True

2. train/val/test 내부 폴더 확인

[train]
클래스로 인식될 폴더:
['crop_img']

[val]
클래스로 인식될 폴더:
['crop_img']

[test]
클래스로 인식될 폴더:
['crop_img']

3. ImageFolder 클래스 매핑 확인
train class_to_idx: {'crop_img': 0}
val   class_to_idx: {'crop_img': 0}
test  class_to_idx: {'crop_img': 0}

train classes: ['crop_img']
val   classes: ['crop_img']
test  classes: ['crop_img']

num_classes
train: 1
val  : 1
test : 1

4. 클래스 매핑 일치 여부
정상: train/val/test 클래스 매핑이 모두 동일함

5. 클래스별 이미지 개수 확인

[train] 총 이미지 수: 14393
crop_img: 14393장

[val] 총 이미지 수: 1777
crop_img: 1777장

[test] 총 이미지 수: 1818
crop_img: 1818장

6. 샘플 이미지 경로 / 라벨 확인

[train] 샘플 10개
label=0 (crop_img) | C:\Users\Konyang\Desktop\cropped_preprocessed_dataset\train\crop_img\C_A2\IMG_C_A2_000003.jpg
lab